# All you need to modify

In [ ]:
import pandas as pd
import os

#zzzzzzzz
#yyyyyyyyy
students_sheets = pd.read_excel("/content/Students_CSD3_Weekly Reports (Yuanjing) - Mar 16 2026.xlsx", sheet_name=None)

## Adults Sheets
adults_sheets = "/content/Adults_CSD3_Weekly Reports (Yuanjing) - Mar 16 2026.xlsx"

## All Sheets
all_sheets = "/content/All_CSD3_Weekly Reports (Yuanjing) - Mar 16 2026.xlsx"


# Student Summary Statistics - Target # of Students Served (don't include Total)
target_values = [150,200,100]




FileNotFoundError: [Errno 2] No such file or directory: '/content/Students_CSD3_Weekly Reports (Yuanjing) - Mar 16 2026.xlsx'

# Code Part (don't need to modify)

## 🍀Student Summary Statistics:


In [ ]:
df_part_by_hour=  students_sheets['Participants By Hour Band']
df_part_by_hour.head()

,Participants By Hour Band,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Participant Count,Participant Count,Participant Count,Hour Band,Hour Band,Hour Band,Hour Band,Hour Band,Hour Band
4,Institution,Site,GradeLevel,0,Less Than 15,15-44,45-89,90-179,180-269


In [ ]:
# Use the 3rd row (index 2) as column names
df_part_by_hour.columns = df_part_by_hour.iloc[4]

# Drop the first three rows and reset index
df_part_by_hour = df_part_by_hour.iloc[5:].reset_index(drop=True)

df_part_by_hour.head(10)

4,Institution,Site,GradeLevel,0,Less Than 15,15-44,45-89,90-179,180-269
0,"Community School District 3, NYCDOE 8026",High School of Arts and Technology,09,18,108,36,3,0,0
1,NaN,NaN,10,17,59,13,5,0,0
2,NaN,NaN,11,19,56,10,3,0,0
3,NaN,NaN,12,6,6,1,0,0,0
4,NaN,Subtotal,Subtotal,60,229,60,11,0,0
5,NaN,P.S. 165 Robert E. Simon,01,1,36,5,5,27,0
6,NaN,NaN,02,0,37,9,6,29,0
7,NaN,NaN,03,0,31,8,11,19,0
8,NaN,NaN,04,0,36,12,3,24,1
9,NaN,NaN,05,0,79,0,1,11,0


In [ ]:
# the average column from Daily Site Attendance Summary
daily_site_att = students_sheets['Daily Site Attendance Summary']

daily_site_att.columns = daily_site_att.iloc[2]  # Set 3rd row as header
daily_site_att = daily_site_att.iloc[3:]          # Drop first 3 rows
daily_site_att.columns.name = None                # Remove column index name
daily_site_att = daily_site_att.reset_index(drop=True)

daily_site_att = daily_site_att[['Total']]        # Keep only Total column

daily_site_att = daily_site_att.iloc[:-1]         # Drop last row
daily_site_att['Total'] = daily_site_att['Total'].str.extract(r'(\d+\.?\d*)')  # Keep number only


In [ ]:
print(df_part_by_hour.columns)

Index(['Institution', 'Site', 'GradeLevel', '0', 'Less Than 15', '15-44',
       '45-89', '90-179', '180-269'],
      dtype='object', name=4)


In [ ]:
all_cols = ['0', 'Less Than 15', '15-44', '45-89', '90-179', '180-269', '270+']
served_cols = ['Less Than 15', '15-44', '45-89', '90-179', '180-269', '270+']
plus15_cols = ['15-44', '45-89', '90-179', '180-269', '270+']
plus90_cols = ['90-179', '180-269', '270+']

# Filter each col list to only include columns that exist in the dataframe
existing_cols = df_part_by_hour.columns.tolist()
all_cols     = [c for c in all_cols     if c in existing_cols]
served_cols  = [c for c in served_cols  if c in existing_cols]
plus15_cols  = [c for c in plus15_cols  if c in existing_cols]
plus90_cols  = [c for c in plus90_cols  if c in existing_cols]

# Convert to numeric — this fixes the TypeError
df_part_by_hour[all_cols] = df_part_by_hour[all_cols].apply(
    pd.to_numeric, errors='coerce'
).fillna(0)  # <-- replaces any non-numeric/NaN with 0 instead of leaving as string

# Filter Subtotal rows
df_sub = df_part_by_hour[df_part_by_hour['Site'] == 'Subtotal']

# Build completely new dataframe
df_totals = pd.DataFrame({
    '[Total # Enrolled]': df_sub[all_cols].sum(axis=1),
    '[Total # Served]':   df_sub[served_cols].sum(axis=1),
    '[Total # 15+]':      df_sub[plus15_cols].sum(axis=1),
    '[Total # 90+]':      df_sub[plus90_cols].sum(axis=1)
})

# Insert as the first column (position 0)
df_totals.insert(0, '[Target # of students served]', target_values)

In [ ]:
## the average column from Daily Site Attendance Summary
daily_site_att['Total'] = daily_site_att['Total'].astype(int)
daily_site_att = daily_site_att.rename(columns={'Total': 'Avg. # of Students Per Day'})
# Insert after [Total # Served] (now at position 2)
df_totals.insert(3, 'Avg. # of Students Per Day', daily_site_att['Avg. # of Students Per Day'].values)
print(df_totals)

    [Target # of students served]  [Total # Enrolled]  \
4                             150                 360   
12                            200                 465   
19                            100                 146   

    Avg. # of Students Per Day  [Total # Served]  [Total # 15+]  [Total # 90+]  
4                           30               300             71              0  
12                         109               464            211            133  
19                          47               145            105             15  


In [ ]:
##INSERT SCHOOL NAMES
# Create an ordered list of valid school names, skipping NaN and 'Subtotal'
school_names = [
    row['Site']
    for _, row in df_part_by_hour.iterrows()
    if pd.notna(row['Site']) and row['Site'] != 'Subtotal' and row['Site'] != 'Total'
]

school_names

# Add the actual Site names as the leftmost column
df_totals.insert(0, 'School', school_names)

# Create the final total row
total_row = pd.DataFrame(df_totals.iloc[:, 1:].sum()).T
total_row.insert(0, 'School', 'Total')

# Append the total row
df_totals = pd.concat([df_totals, total_row], ignore_index=True)

df_totals

,School,[Target # of students served],[Total # Enrolled],Avg. # of Students Per Day,[Total # Served],[Total # 15+],[Total # 90+]
0,High School of Arts and Technology,150,360,30,300,71,0
1,P.S. 165 Robert E. Simon,200,465,109,464,211,133
2,STEM Institute,100,146,47,145,105,15
3,Total,450,971,186,909,387,148


In [ ]:
# Create formatted display columns directly (no helper % columns)

df_totals['# of students 15+ hrs total (% of Target)'] = (
    df_totals['[Total # 15+]'].astype(int).astype(str) +
    " (" +
    (
        (df_totals['[Total # 15+]'] /
         df_totals['[Target # of students served]']) * 100
    ).round().astype(int).astype(str) +
    "%)"
)

df_totals['# of students 90+ hrs total (% of Target)'] = (
    df_totals['[Total # 90+]'].astype(int).astype(str) +
    " (" +
    (
        (df_totals['[Total # 90+]'] /
         df_totals['[Target # of students served]']) * 100
    ).round().astype(int).astype(str) +
    "%)"
)
df_totals = df_totals.drop(columns=['[Total # 15+]', '[Total # 90+]'])
df_totals = df_totals.reset_index(drop=True)
df_totals


,School,[Target # of students served],[Total # Enrolled],Avg. # of Students Per Day,[Total # Served],# of students 15+ hrs total (% of Target),# of students 90+ hrs total (% of Target)
0,High School of Arts and Technology,150,360,30,300,71 (47%),0 (0%)
1,P.S. 165 Robert E. Simon,200,465,109,464,211 (106%),133 (66%)
2,STEM Institute,100,146,47,145,105 (105%),15 (15%)
3,Total,450,971,186,909,387 (86%),148 (33%)


##🌷Family Component Summary Statistics (Last Column)

In [ ]:

# Load the specific sheet for Attendance Hours, skipping the top 2 rows
df_hours = pd.read_excel(adults_sheets, sheet_name="Participant Attendance Hours", skiprows=2)

# Ensure "HoursPresent" is numeric
df_hours["HoursPresent"] = pd.to_numeric(df_hours["HoursPresent"], errors='coerce')

# Clean the ParticipantId: convert to string and remove any trailing '.0' in case pandas loaded it as a float
df_hours["ParticipantId"] = df_hours["ParticipantId"].astype(str).str.replace(r'\.0$', '', regex=True)

# # Filter for rows where HoursPresent is greater than 0
# df_active = df_hours[df_hours["HoursPresent"] > 0]

# Filter for rows where HoursPresent is > 0 AND the ParticipantId length is NOT exactly 9 digits
df_active = df_hours[(df_hours["HoursPresent"] > 0) & (df_hours["ParticipantId"].str.len() != 9)]

# Group by 'Site' and count unique adults using 'ParticipantId'
result = df_active.groupby("Site")["ParticipantId"].nunique().reset_index()

# Rename the column for better presentation
result.rename(columns={"ParticipantId": "Parents Served (Total)"}, inplace=True)

# Calculate the total sum of the 'Parents Served (Total)' column
total_parents = result["Parents Served (Total)"].sum()

# Append the total row to the dataframe (using loc is the cleanest way)
result.loc[len(result)] = {"Site": "Total", "Parents Served (Total)": total_parents}

# Display the final table
display(result)

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Site,Parents Served (Total)
0,High School of Arts and Technology,65
1,P.S. 165 Robert E. Simon,128
2,Total,193


##  🌸Participant Demographics - missing info

In [ ]:
df_part_demo=  students_sheets['Participant Demographics']
df_part_demo.head()

,Participant Demographics,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
0,Participant Demographics,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Institution,Site,Last Name,First Name,ParticipantID,State ParticipantID,Adult,Date Of Birth,School,Grade Level,Gender,Race/Ethnicity,English Learner Status,Lunch Status,Special Education Status,IDEA Disability Type
3,"Community School District 3, NYCDOE 8026",P.S. 165 Robert E. Simon,Aafjes,Lucas,250138685,250138685,No,2020-05-08 00:00:00,P.S. 165 Robert E. Simon,KG,Male,Not Entered,No,Not Entered,No,Not Entered
4,"Community School District 3, NYCDOE 8026",High School of Arts and Technology,Abad Hernandez,Isbeth,232761528,2648790956,No,2008-04-18 00:00:00,High School of Arts and Technology,11,Female,Race and Ethnicity Unknown,No,Full price,No,Not Entered


In [ ]:
# Use the 3rd row (index 2) as column names
df_part_demo.columns = df_part_demo.iloc[2]

# Drop the first three rows and reset index
df_part_demo = df_part_demo.iloc[3:].reset_index(drop=True)

df_part_demo.head()
print(df_part_demo.dtypes)

2
Institution                 object
Site                        object
Last Name                   object
First Name                  object
ParticipantID               object
State ParticipantID         object
Adult                       object
Date Of Birth               object
School                      object
Grade Level                 object
Gender                      object
Race/Ethnicity              object
English Learner Status      object
Lunch Status                object
Special Education Status    object
IDEA Disability Type        object
dtype: object


In [ ]:
def summarize_missing_by_school(df, columns_to_check, category_col="Site"):
    if category_col not in df.columns:
        raise ValueError(f"{category_col} not found in DataFrame columns")

    missing_site_rows = df[df[category_col].isna() | (df[category_col].astype(str).str.strip() == "")].copy()

    subset = df[columns_to_check + [category_col]].copy()
    subset_for_grouping = subset[subset[category_col].notna()].copy()
    subset_for_grouping[category_col] = (
        subset_for_grouping[category_col].astype(str).str.title()
    )

    ## Flagging missing values per col
    for col in columns_to_check:
        cleaned = subset_for_grouping[col].astype(str).str.strip()
        not_entered_mask = cleaned.str.lower() == "not entered"
        if col == "Gender":
            valid_gender = cleaned.str.title().isin(["Male", "Female", "Non-Binary"])
            subset_for_grouping[col + "_missing"] = ((~valid_gender) | not_entered_mask).astype(int)
        else:
            subset_for_grouping[col + "_missing"] = (
                subset_for_grouping[col].isna() | not_entered_mask
            ).astype(int)
## Validating Participant IDs
    pid  = df.loc[subset_for_grouping.index, "ParticipantID"].astype(str).str.strip()
    spid = df.loc[subset_for_grouping.index, "State ParticipantID"].astype(str).str.strip()
    valid_pid  = pid.str.match(r'^[12]\d{8}$')
    valid_spid = spid.str.match(r'^[12]\d{8}$')
    subset_for_grouping["ParticipantID_missing"]       = (~valid_pid).astype(int)
    subset_for_grouping["State ParticipantID_missing"] = (~valid_spid).astype(int)

    valid_osis = (pid == spid) & valid_pid & valid_spid
    subset_for_grouping["OSIS_missing"] = (~valid_osis).astype(int)

    missing_cols = (
        [col + "_missing" for col in columns_to_check]
        + ["ParticipantID_missing", "State ParticipantID_missing", "OSIS_missing"]
    )
    pivot = (
        subset_for_grouping
        .groupby(category_col)[missing_cols]
        .sum()
        .reset_index()
    )
    total_row = pd.DataFrame(pivot[missing_cols].sum()).T
    total_row[category_col] = "Total"
    pivot = pd.concat([pivot, total_row], ignore_index=True)

    all_missing_flags = df.copy()
    pid_all  = all_missing_flags["ParticipantID"].astype(str).str.strip()
    spid_all = all_missing_flags["State ParticipantID"].astype(str).str.strip()
    valid_pid_all  = pid_all.str.match(r'^[12]\d{8}$')
    valid_spid_all = spid_all.str.match(r'^[12]\d{8}$')
    valid_osis_all = (pid_all == spid_all) & valid_pid_all & valid_spid_all

    for col in columns_to_check:
        cleaned = all_missing_flags[col].astype(str).str.strip()
        not_entered_mask = cleaned.str.lower() == "not entered"
        if col == "Gender":
            valid_gender = cleaned.str.title().isin(["Male", "Female", "Non-Binary"])
            all_missing_flags[col + "_missing"] = ((~valid_gender) | not_entered_mask).astype(int)
        else:
            all_missing_flags[col + "_missing"] = (all_missing_flags[col].isna() | not_entered_mask).astype(int)

    all_missing_flags["ParticipantID_missing"]       = (~valid_pid_all).astype(int)
    all_missing_flags["State ParticipantID_missing"] = (~valid_spid_all).astype(int)
    all_missing_flags["OSIS_missing"]                = (~valid_osis_all).astype(int)

    # ← NEW: flag DOBs where year > 2023 (i.e. born after 2023, less than ~3 years old)
    # ✅ Works with datetime objects directly
    dob_parsed = pd.to_datetime(all_missing_flags["Date Of Birth"], errors="coerce")
    all_missing_flags["DOB_too_young"] = (dob_parsed.dt.year > 2023).astype(int)  # ← NEW

    flag_cols = (
        [col + "_missing" for col in columns_to_check]
        + ["ParticipantID_missing"]
    )
    total_missing_rows = all_missing_flags[all_missing_flags[flag_cols].sum(axis=1) > 0].copy()

    # ← NEW: also pull rows where DOB is suspiciously young, even if nothing else is missing
    young_dob_rows = all_missing_flags[all_missing_flags["DOB_too_young"] == 1].copy()  # ← NEW

    return pivot, missing_site_rows, total_missing_rows, flag_cols, young_dob_rows  # ← NEW

In [ ]:
# # Should show 1 row with 2025
# print(young_dob_rows["Date Of Birth"].tolist())

In [ ]:
def apply_missing_highlights(output_filename, total_missing_rows, flag_cols, young_dob_rows):
    red_fill  = PatternFill("solid", start_color="FF9999", end_color="FF9999")
    blue_fill = PatternFill("solid", start_color="9999FF", end_color="9999FF")

    flag_to_original = {
        fc: fc[: -len("_missing")]
        for fc in flag_cols
        if fc.endswith("_missing")
    }

    wb = load_workbook(output_filename)

    # --- Red highlights on Total Missing Info ---
    ws = wb["Total Missing Info"]
    header = {cell.value: cell.column for cell in ws[1]}
    for row_idx, (_, row) in enumerate(total_missing_rows.iterrows(), start=2):
        for flag_col, orig_col in flag_to_original.items():
            if orig_col in header and row.get(flag_col, 0) == 1:
                ws.cell(row=row_idx, column=header[orig_col]).fill = red_fill

    # --- Blue highlights on Young DOB sheet ---
    ws2 = wb["Young DOB"]
    header2 = {cell.value: cell.column for cell in ws2[1]}
    if "Date Of Birth" in header2:
        dob_col_idx = header2["Date Of Birth"]
        for row_idx in range(2, len(young_dob_rows) + 2):
            ws2.cell(row=row_idx, column=dob_col_idx).fill = blue_fill

    wb.save(output_filename)
    print("Highlights applied.")

 call the function here

In [ ]:


columns_of_interest = ["Date Of Birth", "Grade Level", "Gender"]

missing_summary, missing_site_rows, total_missing_rows, flag_cols, young_dob_rows = summarize_missing_by_school(
    df_part_demo, columns_of_interest
)


## 🪻Site Summary Report

In [ ]:
import pandas as pd
import numpy as np
import os



# Load the specific sheets and skip rows accordingly
df_act = pd.read_excel(all_sheets, sheet_name="Activity-Session Details", skiprows=2)
df_enr = pd.read_excel(all_sheets, sheet_name="Session Enrollment by Session", skiprows=2)
# Excel limits sheet names to 31 chars, so 'Summary' is cut off to 'Summa'
df_att = pd.read_excel(all_sheets, sheet_name="Daily Activity Attendance Summa", skiprows=4)

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [ ]:
# 2. Extract and format the Activity-Session Details
cols_act = ["Site", "Activity", "Session", "Days Scheduled", "Session Start Date"]
df_act = df_act[cols_act].copy()
# Convert "Days Scheduled" to numeric, converting errors/blanks to NaN
df_act["Days Scheduled"] = pd.to_numeric(df_act["Days Scheduled"], errors='coerce')
df_act["Session Start Date"] = pd.to_datetime(df_act["Session Start Date"], errors='coerce')
today = pd.Timestamp.today().normalize()

In [ ]:
# 3. Extract and format Session Enrollment by Session
cols_enr = ["Site", "Activity", "Session", "Enrolled Count"]
df_enr = df_enr[cols_enr].copy()
# Convert to numeric and rename the column
df_enr["Enrolled Count"] = pd.to_numeric(df_enr["Enrolled Count"], errors='coerce')
df_enr.rename(columns={"Enrolled Count": "Enrolled Participant"}, inplace=True)

In [ ]:
# 4. Extract and format Daily Activity Attendance Summary
cols_att = ["Site", "Activity", "Session", "Total"]
df_att = df_att[cols_att].copy()

# Function to extract the numeric value from the "Total" column string
def extract_average(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).replace('Average:', '').strip()
    try:
        return float(val_str)
    except ValueError:
        return np.nan
# load the numeric and rename column
df_att["Total"] = df_att["Total"].apply(extract_average)
df_att.rename(columns={"Total": "Average Daily Attendance"}, inplace=True)

In [ ]:
# 5. Get a list of unique, valid sites
sites = df_act['Site'].dropna().unique().tolist()
# Filter out any aggregate rows like 'Total' or empty strings
sites = [s for s in sites if str(s).strip() != '' and not str(s).startswith('Total') and not str(s).startswith('Average')]

# 6. Create a dictionary to hold the merged DataFrame for each site
site_tables = {}

for site in sites:
    # Filter the three dataframes for the current site
    site_act = df_act[df_act['Site'] == site].copy()
    site_enr = df_enr[df_enr['Site'] == site].copy()
    site_att = df_att[df_att['Site'] == site].copy()

    # Merge Activity and Enrollment
    merged_site = pd.merge(site_act, site_enr, on=["Site", "Activity", "Session"], how="outer")

    # Merge the result with Attendance
    merged_site = pd.merge(merged_site, site_att, on=["Site", "Activity", "Session"], how="outer")

    merged_site = merged_site[~(merged_site['Session Start Date'] >= today)]

    # Delete the 'Session Start Date' column from the final table
    if 'Session Start Date' in merged_site.columns:
        merged_site = merged_site.drop(columns=['Session Start Date'])

    # Replace NaN (null) values with "-"
    merged_site = merged_site.fillna("-")

    # merged_site = merged_site.replace(r'^\s*$', '-', regex=True)

    # Store in our dictionary
    site_tables[site] = merged_site.sort_values(by=['Activity', 'Session'], ascending=True).reset_index(drop=True)

In [ ]:
# 7. Print or preview the results
for site_name, final_df in site_tables.items():
    print(f"\n{'='*50}")
    print(f"Site: {site_name}")
    print(f"{'='*50}")
    # Display the first few rows for validation in Colab
    display(final_df.head())


Site: High School of Arts and Technology


,Site,Activity,Session,Days Scheduled,Enrolled Participant,Average Daily Attendance
0,High School of Arts and Technology,Academic Enrichment,Regents Prep - Algebra,6,22.0,8.0
1,High School of Arts and Technology,Academic Enrichment,Regents Prep - Biology,6,5.0,3.0
2,High School of Arts and Technology,Academic Enrichment,Regents Prep - English,6,9.0,4.0
3,High School of Arts and Technology,Academic Enrichment,Regents Prep - Living Environment,5,13.0,6.0
4,High School of Arts and Technology,Academic Enrichment,S3 - Algebra,4,45.0,14.0



Site: P.S. 165 Robert E. Simon


,Site,Activity,Session,Days Scheduled,Enrolled Participant,Average Daily Attendance
0,P.S. 165 Robert E. Simon,Academic Enrichment {ELA & Math Prep},Ela & Math Prep | De La Cruz,9,18.0,-
1,P.S. 165 Robert E. Simon,Academic Enrichment {ELA & Math Prep},Ela & Math Prep | La Guardia,9,6.0,-
2,P.S. 165 Robert E. Simon,Academic Enrichment {ELA & Math Prep},Ela & Math Prep | Martinez,9,5.0,-
3,P.S. 165 Robert E. Simon,Academic Enrichment {ELA & Math Prep},Ela & Math Prep | Ms. Guzman,9,10.0,-
4,P.S. 165 Robert E. Simon,Academic Enrichment {ELA & Math Prep},Ela & Math Prep | Ms. Ortiz,9,6.0,-



Site: STEM Institute


,Site,Activity,Session,Days Scheduled,Enrolled Participant,Average Daily Attendance
0,STEM Institute,Academic Enrichment,ELA/Math Tutoring (Mr. Nesbitt),49,14.0,14.0
1,STEM Institute,Academic Enrichment,ELA/Math Tutoring (Ms. Cabrera),50,13.0,13.0
2,STEM Institute,Academic Enrichment,ELA/Math Tutoring (Ms. Gil),51,13.0,14.0
3,STEM Institute,Academic Enrichment,ELA/Math Tutoring (Ms. Nisiryou/Bratton),51,22.0,21.0
4,STEM Institute,Academic Enrichment,ELA/Math Tutoring (Ms. Thame),49,18.0,17.0





##🌈Full Report Download

In [ ]:
from google.colab import files
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

output_filename = "Processed_Site_Tables.xlsx"

with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    df_totals.to_excel(writer, sheet_name="Student Statistics", index=False)
    print(f"Successfully created sheet for: Student Statistics (Rows: {len(df_totals)})")

    result.to_excel(writer, sheet_name="Parents Served Summary", index=False)
    print(f"Successfully created sheet for: Parents Served Summary (Rows: {len(result)})")

    missing_summary.to_excel(writer, sheet_name="Missing Summary", index=False)
    print(f"Successfully created sheet for: Missing Summary (Rows: {len(missing_summary)})")

    missing_site_rows.to_excel(writer, sheet_name="Missing Site Info", index=False)
    print(f"Successfully created sheet for: Missing Site Info (Rows: {len(missing_site_rows)})")

    display_cols = [c for c in total_missing_rows.columns if not c.endswith("_missing") and c != "DOB_too_young"]
    total_missing_rows[display_cols].to_excel(writer, sheet_name="Total Missing Info", index=False)
    print(f"Successfully created sheet for: Total Missing Info (Rows: {len(total_missing_rows)})")

    young_display_cols = [c for c in young_dob_rows.columns if not c.endswith("_missing") and c != "DOB_too_young"]
    young_dob_rows[young_display_cols].to_excel(writer, sheet_name="Young DOB", index=False)
    print(f"Successfully created sheet for: Young DOB (Rows: {len(young_dob_rows)})")

    for site_name, final_df in site_tables.items():
        safe_sheet_name = (
            str(site_name)[:31]
            .replace(":", "").replace("/", "").replace("\\", "")
            .replace("?", "").replace("*", "")
        )
        final_df.to_excel(writer, sheet_name=safe_sheet_name, index=False)
        print(f"Successfully created sheet for: {safe_sheet_name} (Rows: {len(final_df)})")

# Apply highlights after file is fully closed
apply_missing_highlights(output_filename, total_missing_rows, flag_cols, young_dob_rows)

print(f"\nAll data saved to '{output_filename}'! Triggering download...")
files.download(output_filename)

Successfully created sheet for: Student Statistics (Rows: 4)
Successfully created sheet for: Parents Served Summary (Rows: 3)
Successfully created sheet for: Missing Summary (Rows: 4)
Successfully created sheet for: Missing Site Info (Rows: 0)
Successfully created sheet for: Total Missing Info (Rows: 26)
Successfully created sheet for: Young DOB (Rows: 0)
Successfully created sheet for: High School of Arts and Technol (Rows: 58)
Successfully created sheet for: P.S. 165 Robert E. Simon (Rows: 94)
Successfully created sheet for: STEM Institute (Rows: 22)
Highlights applied.

All data saved to 'Processed_Site_Tables.xlsx'! Triggering download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>